# 01 · Hello, Umbra

The three-line tour of Umbra's open SAR archive: **search** the catalog,
**summarize** what you found, and **see** it as a radar image — no GIS
install, no multi-gigabyte download.

> **Data license.** Umbra open data is **CC BY 4.0**. Anything you publish
> from it must carry *"Contains Umbra open data, licensed under CC BY 4.0."*
> `umbra-py` propagates that string into every derived artifact for you.

This notebook needs the `viz` extra (rasterio + numpy + Pillow +
matplotlib) for the quicklook at the end:

```bash
pip install "umbra-py[viz]"
```

It hits Umbra's public bucket, so run it with network access.

## 1 · Search the catalog

Umbra publishes a *static* STAC catalog with no search API, so there is no
"scenes in this bbox for these dates" endpoint to call. `UmbraCatalog.search`
restores that primitive by crawling the catalog for you and yielding
`UmbraItem`s. We keep this first search tiny and deterministic — one GEC
acquisition — so the notebook runs in seconds.

In [ ]:
from umbra_py import UmbraCatalog

catalog = UmbraCatalog()
items = list(
    catalog.search(
        start="2024-01-01",
        end="2024-12-31",
        product_types=["GEC"],  # geocoded, north-up amplitude GeoTIFF
        max_per_task=1,  # one pass per site keeps the walk short
        limit=1,
    )
)
assert items, "no acquisitions found — is the bucket reachable?"
item = items[0]
item  # UmbraItem renders a footprint + metadata card in Jupyter

## 2 · Summarize it

`summary()` is the human one-paragraph view; `metadata_summary()` is the
same facts as a dict. Every acquisition carries a footprint (`bbox`) and a
set of downloadable product assets.

In [ ]:
print(item.summary())

assert item.id
assert item.bbox is not None
assert item.available_assets, "item has no downloadable assets"

## 3 · It speaks GeoJSON (zero-glue geopandas / shapely)

`UmbraItem` implements Python's `__geo_interface__`, so the whole scientific
geo stack ingests it with no conversion code — `shapely.geometry.shape(item)`,
`geopandas.GeoDataFrame.from_features([item])`, leafmap, and friends all just
work. Here is the raw GeoJSON `Feature`:

In [ ]:
feature = item.to_geojson()
assert feature["type"] == "Feature"
assert feature["geometry"]["type"] in ("Polygon", "MultiPolygon")
print(feature["geometry"]["type"], "with", len(feature["properties"]), "properties")

# If geopandas is installed, the item drops straight into a GeoDataFrame:
try:
    import geopandas as gpd

    gdf = gpd.GeoDataFrame.from_features([item])
    print(gdf[["geometry"]].head())
except ImportError:
    print("(install geopandas to load this footprint into a GeoDataFrame)")

## 4 · The model-ready context card

`to_llm_context()` returns the same scene as an explanation-rich card for
prompting a language model: each product type carries a one-line meaning, the
polarizations carry the change-detection caveat, and the mandatory CC-BY
attribution travels with the data. (This is exactly what the `umbra-mcp`
server hands a model.)

In [ ]:
from umbra_py import ATTRIBUTION

card = item.to_llm_context()
assert card["attribution"] == ATTRIBUTION  # license discipline is automatic
print("products:", [p["type"] for p in card["products"]])
print("attribution:", card["attribution"])

## 5 · See the radar

`save_quicklook` streams a downsampled overview of the cloud-optimized GeoTIFF
over HTTP range requests (no full download), applies a SAR-tuned stretch, and
writes an image. SAR spans an enormous dynamic range, so `db=True` (decibel
scaling) plus a colormap is the most legible look.

In [ ]:
from IPython.display import Image

from umbra_py import save_quicklook

png = save_quicklook(
    item,
    "hello_umbra.png",
    db=True,
    colormap="magma",
    max_size=512,  # small, fast overview
)
assert png.exists()
assert png.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"  # it really is a PNG

Image(filename=str(png))

## Where next

- **`02_download_and_open_gec.ipynb`** — open a scene as an analysis-ready
  `xarray.DataArray`.
- **`03_change_detection.ipynb`** — composite two passes of one site into a
  color change image.
- The CLI mirrors every call here: `umbra search`, `umbra info`,
  `umbra quicklook`.

*Contains Umbra open data, licensed under CC BY 4.0.*